# ISTVT — Training
Trains the ISTVT model on the preprocessed face-crop dataset.

**Key Colab survival features:**
- Checkpoint saved every epoch → auto-resumed on reconnect
- Mixed precision (AMP) for T4 memory efficiency
- Gradient accumulation (effective batch = 16 with batch_size=4)


In [ ]:
# Install dependencies
!pip install scikit-learn -q

In [ ]:
import os, time
import torch
import torch.nn as nn
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from sklearn.metrics import roc_auc_score

# ── import from other notebooks (run those cells first, or paste classes here)
# from model   import ISTVT
# from dataset import make_loaders

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

## Config — Edit These

In [ ]:
# ── CONFIGURE ──────────────────────────────────────────────────
DATA_ROOT = "/kaggle/working/istvt_data"
CKPT_DIR  = "/kaggle/working/checkpoints"
EPOCHS           = 20
BATCH_SIZE       = 4
GRAD_ACCUM_STEPS = 4       # effective batch = 4 * 4 = 16
LR               = 1e-4
IMG_SIZE         = 128
SEQ_LEN          = 6
EMBED_DIM        = 256
DEPTH            = 6
NUM_HEADS        = 8
NUM_WORKERS      = 2
# ───────────────────────────────────────────────────────────────

## Evaluation Helper

In [ ]:
def evaluate(model, loader, device):
    model.eval()
    criterion = nn.BCEWithLogitsLoss()
    all_logits, all_labels, total_loss = [], [], 0.0

    with torch.no_grad():
        for seq, label in loader:
            seq, label = seq.to(device), label.to(device)
            logits = model(seq)
            total_loss += criterion(logits, label).item() * seq.size(0)
            all_logits.append(logits.cpu())
            all_labels.append(label.cpu())

    all_logits = torch.cat(all_logits).squeeze(-1)
    all_labels = torch.cat(all_labels).squeeze(-1)
    preds = (torch.sigmoid(all_logits) > 0.5).float()
    acc   = (preds == all_labels).float().mean().item()
    try:
        auc = roc_auc_score(all_labels.numpy(), torch.sigmoid(all_logits).numpy())
    except ValueError:
        auc = float("nan")
    return total_loss / len(loader.dataset), acc, auc

print("evaluate defined ✓")

## Training Loop

In [ ]:
os.makedirs(CKPT_DIR, exist_ok=True)

train_loader, val_loader = make_loaders(DATA_ROOT, BATCH_SIZE, SEQ_LEN, IMG_SIZE, NUM_WORKERS)
print(f"Train: {len(train_loader.dataset)} | Val: {len(val_loader.dataset)}")

model     = ISTVT(IMG_SIZE, SEQ_LEN, EMBED_DIM, DEPTH, NUM_HEADS).to(device)
optimizer = AdamW(model.parameters(), lr=LR, weight_decay=0.01)
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS)
criterion = nn.BCEWithLogitsLoss()
scaler    = torch.cuda.amp.GradScaler(enabled=(device == "cuda"))

# ── Auto-resume from last checkpoint ────────────────────────────────────────
start_epoch, best_auc = 0, 0.0
ckpt_last = os.path.join(CKPT_DIR, "last.pt")
ckpt_best = os.path.join(CKPT_DIR, "best.pt")

if os.path.exists(ckpt_last):
    print(f"Resuming from {ckpt_last}")
    state = torch.load(ckpt_last, map_location=device)
    model.load_state_dict(state["model"])
    optimizer.load_state_dict(state["optimizer"])
    scheduler.load_state_dict(state["scheduler"])
    start_epoch = state["epoch"] + 1
    best_auc    = state.get("best_auc", 0.0)
    print(f"Resumed at epoch {start_epoch}, best AUROC so far: {best_auc:.4f}")

# ── Training ─────────────────────────────────────────────────────────────────
for epoch in range(start_epoch, EPOCHS):
    model.train()
    running_loss = 0.0
    optimizer.zero_grad()
    t0 = time.time()

    for step, (seq, label) in enumerate(train_loader):
        seq, label = seq.to(device), label.to(device)
        with torch.cuda.amp.autocast(enabled=(device == "cuda")):
            loss = criterion(model(seq), label) / GRAD_ACCUM_STEPS

        scaler.scale(loss).backward()
        running_loss += loss.item() * GRAD_ACCUM_STEPS * seq.size(0)

        if (step + 1) % GRAD_ACCUM_STEPS == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()

    scheduler.step()
    train_loss = running_loss / len(train_loader.dataset)
    val_loss, val_acc, val_auc = evaluate(model, val_loader, device)
    dt = time.time() - t0

    print(f"Epoch {epoch+1:02d}/{EPOCHS} | "
          f"train_loss={train_loss:.4f}  val_loss={val_loss:.4f}  "
          f"val_acc={val_acc:.4f}  val_auc={val_auc:.4f}  ({dt:.1f}s)")

    # Save checkpoint every epoch
    state = {"model": model.state_dict(), "optimizer": optimizer.state_dict(),
             "scheduler": scheduler.state_dict(), "epoch": epoch, "best_auc": best_auc}
    torch.save(state, ckpt_last)

    if val_auc > best_auc:
        best_auc = val_auc
        state["best_auc"] = best_auc
        torch.save(state, ckpt_best)
        print(f"  → New best saved (AUROC={best_auc:.4f})")

print("Training complete ✓")